# Provide the API key

- If you don’t have an API key, go to [Groq Console](https://console.groq.com/keys) and create an account (or log in if you already have one).
- Once logged in, click “**Create API Key”** to generate your key. Add any name to it and click the **Submit** button. 
- Copy the generated API key and paste it below.

In [ ]:
API = "Copy_and_paste_your_API_key_here"

# Install Required Libraries and spaCy Laguage Model
**※ First time only**

In [ ]:
!pip install groq pandas spacy matplotlib numpy

In [ ]:
!python -m spacy download en_core_web_md

# Read All Functions

In [40]:
import pandas as pd
from groq import Groq
import difflib
import re
import spacy
import os


nlp = spacy.load('en_core_web_md') # en_core_web_sm
nlp.max_length = 10000000


def normalize_text(text):
    text = re.sub(r"\s\s+", " ", text)
    text = text.replace("’", "'")
    text = text.replace("‘", "'")
    text = text.replace("“", '"')
    text = text.replace("”", '"')
    text = text.replace("‐", "-")
    text = text.replace("？", "?")
    text = re.sub(r'[^\x00-\x7F]+', '', text)
    return text


def safe_division(n, d):
    return n / d if d else 0


def extract_t_units(text):
    doc = nlp(text)
    t_units = []
    current_unit = []
    fanboys = ["and", "nor", "but", "or", "yet", "so"]
    conjunctive_adverbs = ["consequently", "therefore", "however", "moreover", "furthermore", "nevertheless", "thus", "hence", "still", "instead", "otherwise", "likewise", "rather", "accordingly", "besides", "alternatively"]
    
    def is_independent_clause(span):
        has_subject = any(token.dep_ in ["nsubj", "nsubjpass"] for token in span)
        has_verb = any(token.pos_ in ["VERB", "AUX"] for token in span)
        return has_subject and has_verb
    
    def shares_same_subject(token):
        if token.dep_ == "conj" and token.head.pos_ == "VERB":
            head_verb = token.head
            for child in head_verb.children:
                if child.dep_ == "nsubj":
                    return True
        return False
    
    def find_next_clause_start(i):
        j = i
        while j < len(doc) and (doc[j].text.lower() in conjunctive_adverbs or
                               doc[j].text in [",", ";"]):
            j += 1
        return j
    
    def should_split_and(token, i):
        if token.text == ";" or (token.text.lower() in conjunctive_adverbs and i > 0 and doc[i-1].text in [";", ","]):
            next_idx = find_next_clause_start(i + 1)
            if next_idx < len(doc):
                next_clause = doc[next_idx:]
                return is_independent_clause(next_clause)
            return False
            
        if token.text.lower() in fanboys and i + 1 < len(doc):
            next_token = doc[i + 1]
            
            # Check if we're in a relative clause
            if i > 0:
                prev_token = doc[i-1]
                if (prev_token.dep_ == "relcl" or
                    (prev_token.dep_ == "nsubj" and prev_token.text.lower() == "who") or
                    token.head.dep_ == "relcl"):
                    return False
            
            if token.dep_ == "cc":
                head = token.head
                if head.tag_ in ["VBG", "TO", "NN", "NNS"]:
                    return False
                if head.dep_ in ["pcomp", "conj", "dobj", "relcl"]:
                    return False
            
            j = i + 1
            while j < len(doc) and doc[j].pos_ not in ["VERB", "AUX", "NOUN", "PRON"]:
                j += 1
    
            if j < len(doc):
                next_verb_or_noun = doc[j]
                
                if next_verb_or_noun.dep_ == "conj":
                    return False
                
                if any(t.dep_ == "xcomp" for t in next_verb_or_noun.children):
                    return False
                
                # Check for relative clauses
                if next_verb_or_noun.dep_ == "relcl" or (j > 0 and doc[j-1].text.lower() == "who"):
                    return False
                
                if next_verb_or_noun.dep_ in ["nsubj", "nsubjpass"]:
                    # Don't split if we're in a relative clause
                    if any(t.dep_ == "relcl" for t in doc[i:j+1]):
                        return False
                    return True
                
                next_clause = doc[j:]
                return is_independent_clause(next_clause)
    
        return False
    
    split_indices = [0]
    split_pairs = []
    
    for i, token in enumerate(doc):
        if (token.text == ";" or 
            (token.text.lower() in conjunctive_adverbs and i > 0 and doc[i-1].text in [";", ","]) or
            token.text.lower() in fanboys):
            
            if i < len(doc) - 1:
                if should_split_and(token, i):
                    if token.text == ";":
                        split_pairs.append((split_indices[-1], i))
                        split_indices.append(i)
                    else:
                        if i > 0 and doc[i-1].text == ";":
                            split_pairs.append((split_indices[-1], i-1))
                            split_indices.append(i-1)
                        else:
                            split_pairs.append((split_indices[-1], i))
                            split_indices.append(i)
    
    split_pairs.append((split_indices[-1], len(doc)))
    
    for start, end in split_pairs:
        unit_text = ''.join(doc[j].text_with_ws for j in range(start, end))
        if unit_text.strip():
            t_units.append(unit_text.strip())
    
    return t_units


def merge_punctuation(lst):
    if len(lst) < 2:
        return lst
    last_element = lst[-1].strip()
    if last_element in [".", ",", "!", "?", '"', "'"]:
        lst[-2] = lst[-2].rstrip() + last_element
        lst.pop()
    return lst


def is_permissive_construction(token):
    """Returns True if the token is part of a permissive or causative construction."""
    # Check if the token is a complement (ccomp or xcomp) and its head is a controlling verb
    if token.dep_ in ["ccomp", "xcomp"] and token.head.pos_ == "VERB":
        # Check for common causative or permissive structures
        if token.head.lemma_ in ["allow", "let", "help", "make", "have", "get", "want"]:
            return True
        # More generally, if it's in a complement clause controlled by another verb
        if token.head.dep_ in ["ROOT", "ccomp", "xcomp"]:
            return True
    return False


def extract_verbs(doc):
    clause_verbs = []

    for token in doc:
        # Skip if it's a gerund modifier or an xcomp
        if token.dep_ in ["xcomp", "amod"] and token.tag_ == "VBG":
            continue

        # Process verbs, including acl (adjectival clauses)
        if token.pos_ in ["VERB", "AUX"] and token.dep_ not in ["aux"]:
            # Handle infinitive clauses (skip "to" phrases)
            if any(child.dep_ == "aux" and child.pos_ == "PART" and child.text == "to" for child in token.children):
                continue

            # Handle gerunds (skip gerunds that don't introduce clauses)
            if token.tag_ == "VBG" and token.dep_ not in ["advcl", "acl"]:
                continue

            # Check for passive and active constructions
            passive = any(child.dep_ == "auxpass" for child in token.children)
            active = any(child.dep_ in ["nsubj", "csubj", "nsubjpass", "expl"] for child in token.children)

            # Include verbs if there's a clear subject, root verb, or adjectival clause (acl)
            if active or token.dep_ == "ROOT" or passive or token.dep_ == "advcl" or token.dep_ == "acl":
                clause_verbs.append(token)

        # Detect and append verbs in relative clauses
        if token.dep_ == "relcl" and token.head.pos_ == "NOUN":
            if token not in clause_verbs:
                clause_verbs.append(token)

    return clause_verbs


def extract_clauses_using_verbs(doc, clause_verbs, extracted_clauses):
    clauses = []
    skip_until = -1

    for verb in clause_verbs:
        clause_tokens = list(verb.subtree)

        # Check if the verb is part of a coordinated structure (e.g., "analyzes and makes")
        if verb.dep_ == "conj" and verb.head.pos_ == "VERB":
            # Find the coordinating conjunction (e.g., "and")
            for child in verb.head.children:
                if child.dep_ == "cc" and child.pos_ == "CCONJ":
                    clause_tokens = [child] + clause_tokens  # Prepend the conjunction
                    break

        # Ensure that punctuation directly following the clause is included
        if clause_tokens[-1].i + 1 < len(doc) and doc[clause_tokens[-1].i + 1].text in [",", ";", "."]:
            clause_tokens.append(doc[clause_tokens[-1].i + 1])
            skip_until = clause_tokens[-1].i

        # Use token.text_with_ws to preserve spacing
        clause_text = "".join([t.text_with_ws for t in clause_tokens]).strip()
        
        # Check if the clause is not already added to prevent duplication
        if clause_text not in extracted_clauses:
            clauses.append(clause_text)

    return clauses


# Refine clauses using regex rules and string replacements
def refine_clauses(clauses):
    # Apply the first refinement (remove spaces before punctuation and fix contractions)
    clauses_refined = [re.sub(r'\s+([,.;?!])', r'\1', clause).strip() for clause in clauses]
    
    for i in range(len(clauses_refined)):
        # Handle negations like "did n't" -> "didn't"
        clauses_refined[i] = re.sub(r"\b(\w+)\s+n\s*['’]t", r"\1n't", clauses_refined[i])
        # Handle contractions like "I 'm" -> "I'm"
        clauses_refined[i] = re.sub(r"\b(I|you|we|they|he|she|it)\s+['’](m|ve|re|d|ll|s)", r"\1'\2", clauses_refined[i])

    # Additional rules for refining clauses
    refined_clauses = []
    for sentence in clauses_refined:
        # Handle possessives and contractions
        sentence = sentence.replace(" ’s", "’s").replace(" 's", "'s").replace(" ’", "'").replace(" '", "'")
        # Handle quotes and parentheses
        sentence = sentence.replace('“ ', '“').replace(' ”', '”').replace('( ', '(').replace(' )', ')')
        # Handle hyphens
        sentence = sentence.replace(' - ', '-')
        refined_clauses.append(sentence)
        
    return refined_clauses


# Add missing parts of a clause
def add_missing_part(text, output):
    if output:
        first_output = output[0].replace(' ,', ',')
        if first_output in text:
            missing_part = text[:text.index(first_output)].strip()
            output[0] = missing_part + ' ' + output[0].rstrip()

    return output


def compare_and_refine(list_of_sentences):
    refined_list = []
    
    if not list_of_sentences:
        return refined_list
        
    refined_list.append(list_of_sentences[0].strip())
    
    for i in range(1, len(list_of_sentences)):
        current_sentence = list_of_sentences[i].strip()
        prev_sentence = refined_list[-1]

        if prev_sentence in current_sentence:
            remaining_part = current_sentence[len(prev_sentence):].strip()
            if remaining_part:
                refined_list.append(remaining_part)
        elif current_sentence in prev_sentence:
            overlap_start = prev_sentence.find(current_sentence)
            refined_part = prev_sentence[:overlap_start].strip()
            if refined_part:
                refined_list[-1] = refined_part
            refined_list.append(current_sentence)
        else:
            refined_list.append(current_sentence)
    
    return refined_list


def process_all_sentences(clauses_only):
    results = []
    for sentence_list in clauses_only:
        refined = compare_and_refine(sentence_list)
        results.append(refined)
    return results


def clausal_complexity(token, feature_dict):
    if token.pos_ == "VERB":
        if token.dep_ != "aux":
            feature_dict["all_clauses"] += 1
            deps = [child.dep_ for child in token.children]
            if "nsubj" in deps or "nsubjpass" in deps:
                feature_dict["finite_clause"] += 1
                if token.dep_ in ["ROOT", "conj"]:
                    feature_dict["finite_ind_clause"] += 1
                else:
                    feature_dict["finite_dep_clause"] += 1
                if token.dep_ == "ccomp":
                    feature_dict["finite_compl_clause"] += 1
                if token.dep_ == "relcl":
                    feature_dict["finite_relative_clause"] += 1
            else:
                feature_dict["nonfinite_clause"] += 1
            feature_dict["vp_deps"] += len(deps)


def calculate_vp_deps(text):
    doc = nlp(text)
    vd_dict = {"vp_deps": 0, "all_clauses": 0, "finite_clause": 0, "finite_ind_clause": 0, 
                    "finite_dep_clause": 0, "finite_compl_clause": 0, "finite_relative_clause": 0, "nonfinite_clause": 0}

    for token in doc:
        if token.pos_ == 'VERB':
            clausal_complexity(token, vd_dict)

    return vd_dict


def noun_phrase_complexity(token, nominal_dict):
    if token.pos_ == "NOUN":  # only consider common nouns (exclude pronouns and proper nouns)
        nominal_dict["np"] += 1
        deps = [child.dep_ for child in token.children]
        nominal_dict["np_deps"] += len(deps)
        for x in deps:
            if x == "relcl":
                nominal_dict["relcl_dep"] += 1
            if x == "amod":
                nominal_dict["amod_dep"] += 1
            if x == "det":
                nominal_dict["det_dep"] += 1
            if x == "prep":
                nominal_dict["prep_dep"] += 1
            if x == "poss":
                nominal_dict["poss_dep"] += 1
            if x == "cc":
                nominal_dict["cc_dep"] += 1


def calculate_nominal_deps(text):
    doc = nlp(text)
    nominal_dict = {"np": 0, "np_deps": 0, "relcl_dep": 0, "amod_dep": 0, "det_dep": 0, "prep_dep": 0, "poss_dep": 0, "cc_dep": 0}

    for token in doc:
        noun_phrase_complexity(token, nominal_dict)

    return nominal_dict
   

def calculate_errors(original, corrected):
    original_words = original.split()
    corrected_words = corrected.split()
    matcher = difflib.SequenceMatcher(None, original_words, corrected_words)
    errors = 0
    highlighted_original = []
    highlighted_corrected = []
    for opcode in matcher.get_opcodes():
        tag, i1, i2, j1, j2 = opcode
        if tag == 'equal':
            highlighted_original.extend(original_words[i1:i2])
            highlighted_corrected.extend(corrected_words[j1:j2])
        elif tag == 'replace':
            highlighted_original.append('**{}**'.format(' '.join(original_words[i1:i2])))
            highlighted_corrected.append('**{}**'.format(' '.join(corrected_words[j1:j2])))
            errors += 1
        elif tag == 'insert':
            highlighted_corrected.append('**{}**'.format(' '.join(corrected_words[j1:j2])))
            errors += 1
        elif tag == 'delete':
            highlighted_original.append('**{}**'.format(' '.join(original_words[i1:i2])))
            errors += 1
    return errors, ' '.join(highlighted_original), ' '.join(highlighted_corrected)


def correct_sentences(data):
    client = Groq(api_key=API)
    prompt = "Reply with a corrected version of the input sentence with all grammatical, spelling, and punctuation errors fixed. Be strict about the possible errors. If there are no errors, reply with a copy of the original sentence. Please do not add any unnecessary explanations."

    for index, row in data.iterrows():
        text = row['Sentence']
        chat_completion = client.chat.completions.create(
            messages=[
                {"role": "system", "content": prompt},
                {"role": "user", "content": text}
            ],
            model="llama-3.3-70b-versatile",
            temperature=0,       
        )
        response = chat_completion.choices[0].message.content

        error_count, original_highlighted, corrected_highlighted = calculate_errors(text, response)

        data.loc[index, 'Corrected'] = response
        data.loc[index, 'Highlighted Original'] = original_highlighted
        data.loc[index, 'Highlighted Corrected'] = corrected_highlighted
        data.loc[index, 'ErrorCounts'] = error_count

        #time.sleep(1)

    return data


def check_error_in_t_units(corrected_list, t_units_list):
    error_report = []
    t_units_counts = []
    error_free_t_unit = []
    
    for i, t_units in enumerate(t_units_list):
        corrected_sentence = corrected_list[i]
        t_unit_errors = []
        no_error_count = 0
        
        for t_unit in t_units:
            if t_unit.strip() not in corrected_sentence:
                t_unit_errors.append((t_unit, "Error"))
            else:
                t_unit_errors.append((t_unit, "No Error"))
                no_error_count += 1
        
        error_report.append(t_unit_errors)
        t_units_counts.append(len(t_units))
        error_free_t_unit.append(no_error_count)

    return error_report, t_units_counts, error_free_t_unit


def check_errors_in_t_units_and_clauses(corrected_list, t_units_list, clauses_refined):
    t_unit_error_report = []
    clause_error_report = []
    t_units_counts = []
    clauses_counts = []
    error_free_t_unit = []
    error_free_clause = []
    
    for i in range(len(corrected_list)):
        corrected_sentence = corrected_list[i]
        
        t_units = t_units_list[i]
        t_unit_errors = []
        no_error_t_units_count = 0
        
        for t_unit in t_units:
            if t_unit.strip() not in corrected_sentence:
                t_unit_errors.append((t_unit, "Error"))
            else:
                t_unit_errors.append((t_unit, "No Error"))
                no_error_t_units_count += 1
        
        t_unit_error_report.append(t_unit_errors)
        t_units_counts.append(len(t_units))
        error_free_t_unit.append(no_error_t_units_count)
        
        clauses = clauses_refined[i]
        clause_errors = []
        no_error_clauses_count = 0
        
        for clause in clauses:
            if clause.strip() not in corrected_sentence:
                clause_errors.append((clause, "Error"))
            else:
                clause_errors.append((clause, "No Error"))
                no_error_clauses_count += 1
        
        clause_error_report.append(clause_errors)
        clauses_counts.append(len(clauses))
        error_free_clause.append(no_error_clauses_count)
    
    return {
        't_unit_error_report': t_unit_error_report,
        'clause_error_report': clause_error_report,
        't_units_counts': t_units_counts,
        'clauses_counts': clauses_counts,
        'error_free_t_unit': error_free_t_unit,
        'error_free_clause': error_free_clause
    }


# Batch Process

In [42]:
import glob
import time

# Specify the directory containing text files
directory_path = './CAF_analyzer/*.txt' #'./CAF_analyzer/*.txt'
files = glob.glob(directory_path)

# Initialize a DataFrame to store results
results_df = pd.DataFrame(columns=[
    'File Name', 'Total Words', 'Number of Types', 'TTR', 'Number of T-units', 
    'Number of Clauses', 'MLC', 'MLT', 'C/T', 'DC/T', 'Mean Verbal Dependents', 'Mean Nominal Dependents',
    'Number of Errors', "Errors per 100 words", "Errors per Total Words", "Errors per Sentence", "Errors per T-Unit",
    "Errors per Clause", "Error-Free Sentences", "Error-Free T-Units", "Error-Free Clauses", 
    "Error-Free Sentences / Total Sentences", "Error-Free T-Units / Total T-units", "Error-Free Clauses / Total Clauses"
])

# Process each file
print(f"Total files to process: {len(files)}")
for idx, file_path in enumerate(files):
    file_name = os.path.splitext(os.path.basename(file_path))[0]
    print(f"Processing file {idx+1}/{len(files)}: {file_name}")
    
    with open(file_path, 'r', encoding='utf-8') as file:
        text = file.read()
    
    # Normalize and process text
    text = normalize_text(text)
    doc = nlp(text)

    # Count sentences and words
    num_sentences = len(list(doc.sents))
    total_words = sum(1 for token in doc if token.pos_ not in ["PUNCT", "SYM", "SPACE", "X"])
    
    # Token analysis
    tokens = [token.text.lower() for token in doc if not token.is_punct and not token.is_space]
    type_count = len(set(tokens))
    ttr = safe_division(type_count, total_words)

    # T-unit analysis
    sentslist = [sentence.text for sentence in doc.sents]
    t_units_list = []
    for i in sentslist:
        t_units = extract_t_units(i)
        t_units_list.append(t_units)

    total_t_units = sum(len(t) for t in t_units_list)

    # Clausal analysis
    clausal_data = []
    for i in sentslist:
        doc = nlp(i)
        clause_verbs = extract_verbs(doc)
        clausal_data.append({"sentence": i, "total_clauses": len(clause_verbs), "clause_verbs": clause_verbs})

    num_clauses = sum(item['total_clauses'] for item in clausal_data)
    mlc = safe_division(total_words, num_clauses)
    mlt = safe_division(total_words, total_t_units)
    c_t = safe_division(num_clauses, total_t_units)

    joined_string = " ".join(sentslist)
    vd_features = calculate_vp_deps(joined_string)
    dc_t = format(safe_division(vd_features["finite_dep_clause"], total_t_units), ".2f")
    
    mvd = format(safe_division(vd_features["vp_deps"], vd_features["finite_clause"]), ".2f")
    
    nominal_features = calculate_nominal_deps(joined_string)
    mnd = format(safe_division(nominal_features["np_deps"], nominal_features["np"]), ".2f")

    
    matched_t_units = []
    for t_unit, clause_info in zip(t_units_list, clausal_data):
        if len(t_unit) == clause_info['total_clauses']:
            matched_t_units.append(t_unit)
        elif clause_info['total_clauses'] == 0:
            matched_t_units.append(t_unit)        
        else:
            doc = nlp(clause_info['sentence'])
            clause_verbs = extract_verbs(doc)
            extracted_clauses = []
            clauses = extract_clauses_using_verbs(doc, clause_verbs, extracted_clauses)
            clauses_refined = refine_clauses(clauses)
            output_with_missing_part = add_missing_part(clause_info['sentence'], clauses_refined)
            output_with_missing_part = refine_clauses(output_with_missing_part)
    
            if len(output_with_missing_part) == clause_info['total_clauses']:
                matched_t_units.append(output_with_missing_part)
    
    clauses_only = []
    for t_unit in matched_t_units:
        clauses_only.append(t_unit)

    clauses_refined = process_all_sentences(clauses_only)


    for i in range(len(clauses_refined)):
        for j in range(len(clauses_refined[i]) - 1):
            doc = nlp(clauses_refined[i][j])
            last_token = doc[-1]
            
            # Process conjunctions like "and" and "but"
            if last_token.text.lower() in ["and", "but"] and last_token.pos_ == "CCONJ":
                clauses_refined[i][j] = ''.join([token.text_with_ws for token in doc[:-1]]).strip()
                clauses_refined[i][j + 1] = f"{last_token.text} {clauses_refined[i][j + 1]}"
    
            # Adding 'as' processing, if 'as' is at the end
            elif last_token.text.lower() == "as":
                clauses_refined[i][j] = ''.join([token.text_with_ws for token in doc[:-1]]).strip()
                if j + 1 < len(clauses_refined[i]):
                    clauses_refined[i][j + 1] = f"as {clauses_refined[i][j + 1]}"
                elif i + 1 < len(clauses_refined):
                    clauses_refined[i + 1].insert(0, f"as {clauses_refined[i + 1][0]}")
    
    # Process the final clauses_refined output without extra spaces before commas
    clauses_refined = [[clause.replace(' ,', ',').replace(' .', '.').replace(' ?', '?').replace(' !', '!') for clause in sublist] for sublist in clauses_refined]

    data = pd.DataFrame(sentslist, columns=['Sentence'])
    corrected_data = correct_sentences(data)
    corrected_data['ErrorCounts'] = corrected_data['ErrorCounts'].astype(int)
    
    corrected_list = corrected_data['Corrected'].tolist()
    result = check_errors_in_t_units_and_clauses(corrected_list, t_units_list, clauses_refined)
    
    clause_verbs_list = [entry['clause_verbs'] for entry in clausal_data]

    corrected_data['T-unit counts'] = result['t_units_counts']
    corrected_data['Error-free T-units'] = result['error_free_t_unit']
    corrected_data['Clause counts'] = result['clauses_counts']
    corrected_data['Error-free clauses'] = result['error_free_clause']
    corrected_data['Extracted T-units'] = t_units_list
    corrected_data['Extracted clauses'] = clauses_refined
    corrected_data['Clause signaling verbs'] = clause_verbs_list


    total_clauses = corrected_data['Clause counts'].sum()
    errorfree_sent_count = (corrected_data['ErrorCounts'] == 0).sum()
    errorfree_t_unit_count = corrected_data['Error-free T-units'].sum()
    errorfree_clause_count = corrected_data['Error-free clauses'].sum()

    # (1) Total Errors
    total_errors = corrected_data['ErrorCounts'].sum()
    # (2) Errors per 100 Words
    errors_per_hundred_words = (total_errors / total_words) * 100
    formatted_errors = format(errors_per_hundred_words, ".2f")
    # (3) Errors per Total Words
    number_of_errors_per_words = format(safe_division(total_errors, total_words), ".2f")
    # (4) Errors per sentences (errors / num_sentences)
    errors_per_sentence = format(safe_division(total_errors, num_sentences), ".2f")
    # (5) Errors per T-unit
    number_of_errors_per_t_unit = format(safe_division(total_errors, total_t_units), ".2f")
    # (6) Errors per Clause
    number_of_errors_per_clause = format(safe_division(total_errors, total_clauses), ".2f")
    # (7) Error-Free Sentences
    error_free_sentence = errorfree_sent_count
    # (8) Error-free T-units
    error_free_t_unit = errorfree_t_unit_count
    # (9) Error-free Clauses
    error_free_clause = errorfree_clause_count
    # (10) Error-Free Sentences / Sentences
    error_free_sentence_ratio = format(round(safe_division(errorfree_sent_count, num_sentences), 2), ".2f")
    # (11) Error-Free T-units / T-units
    error_free_t_unit_ratio = format(round(safe_division(errorfree_t_unit_count, total_t_units), 2), ".2f")
    # (12) Error-Free Clauses / Clauses
    error_free_clause_ratio = format(round(safe_division(errorfree_clause_count, total_clauses), 2), ".2f")

    # Append results to DataFrame
    new_row = {
        'File Name': file_name,
        'Total Words': total_words,
        'Number of Types': type_count,
        'TTR': ttr,
        'Number of T-units': total_t_units,
        'Number of Clauses': num_clauses,
        'MLC': mlc,
        'MLT': mlt,
        'C/T': c_t,
        'DC/T': dc_t, 
        'Mean Verbal Dependents': mvd,
        'Mean Nominal Dependents': mnd,
        'Number of Errors': total_errors,
        "Errors per 100 words": formatted_errors,
        "Errors per Total Words": number_of_errors_per_words, 
        "Errors per Sentence": errors_per_sentence,
        "Errors per T-Unit": number_of_errors_per_t_unit, 
        "Errors per Clause": number_of_errors_per_clause, 
        "Error-Free Sentences": error_free_sentence,
        "Error-Free T-Units": error_free_t_unit, 
        "Error-Free Clauses": error_free_clause, 
        "Error-Free Sentences / Total Sentences": error_free_sentence_ratio,
        "Error-Free T-Units / Total T-units": error_free_t_unit_ratio,
        "Error-Free Clauses / Total Clauses": error_free_clause_ratio        
    }
    
    # Append the new row to the DataFrame
    results_df = pd.concat([results_df, pd.DataFrame([new_row])], ignore_index=True)
    print(f"Completed processing file: {file_name}")

    time.sleep(1)

print("All files have been processed successfully.")

Total files to process: 100
Processing file 1/100: S0114


/var/folders/r3/g533s7j50k9dc88j5ph06y800000gn/T/ipykernel_50213/177649563.py:194: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results_df = pd.concat([results_df, pd.DataFrame([new_row])], ignore_index=True)


Completed processing file: S0114
Processing file 2/100: S0100
Completed processing file: S0100
Processing file 3/100: S0060
Completed processing file: S0060
Processing file 4/100: S0049
Completed processing file: S0049
Processing file 5/100: S0075
Completed processing file: S0075
Processing file 6/100: S0103
Completed processing file: S0103
Processing file 7/100: S0117
Completed processing file: S0117
Processing file 8/100: S0088
Completed processing file: S0088
Processing file 9/100: S0077
Completed processing file: S0077
Processing file 10/100: S0063
Completed processing file: S0063
Processing file 11/100: S0062
Completed processing file: S0062
Processing file 12/100: S0076
Completed processing file: S0076
Processing file 13/100: S0116
Completed processing file: S0116
Processing file 14/100: S0099
Completed processing file: S0099
Processing file 15/100: S0072
Completed processing file: S0072
Processing file 16/100: S0067
Completed processing file: S0067
Processing file 17/100: S0073


In [44]:
results_df

,File Name,Total Words,Number of Types,TTR,Number of T-units,Number of Clauses,MLC,MLT,C/T,DC/T,...,Errors per Total Words,Errors per Sentence,Errors per T-Unit,Errors per Clause,Error-Free Sentences,Error-Free T-Units,Error-Free Clauses,Error-Free Sentences / Total Sentences,Error-Free T-Units / Total T-units,Error-Free Clauses / Total Clauses
0,S0114,209,109,0.521531,16,23,9.086957,13.062500,1.437500,0.44,...,0.13,1.75,1.75,1.22,3,3,7,0.19,0.19,0.30
1,S0100,208,116,0.557692,14,30,6.933333,14.857143,2.142857,0.86,...,0.09,1.58,1.36,0.63,5,5,19,0.42,0.36,0.63
2,S0060,204,107,0.524510,21,26,7.846154,9.714286,1.238095,0.29,...,0.09,0.90,0.86,0.69,8,9,13,0.40,0.43,0.50
3,S0049,232,121,0.521552,18,27,8.592593,12.888889,1.500000,0.44,...,0.16,3.00,2.00,1.33,1,2,9,0.08,0.11,0.33
4,S0075,236,110,0.466102,21,32,7.375000,11.238095,1.523810,0.52,...,0.07,0.89,0.81,0.53,7,10,20,0.37,0.48,0.62
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,S0093,280,107,0.382143,15,31,9.032258,18.666667,2.066667,1.07,...,0.08,1.53,1.53,0.74,3,3,16,0.20,0.20,0.52
96,S0051,232,126,0.543103,23,29,8.000000,10.086957,1.260870,0.30,...,0.08,1.12,0.83,0.63,6,10,18,0.35,0.43,0.60
97,S0092,214,118,0.551402,14,33,6.484848,15.285714,2.357143,1.00,...,0.10,1.69,1.57,0.67,1,2,15,0.08,0.14,0.45
98,S0086,214,92,0.429907,16,30,7.133333,13.375000,1.875000,0.69,...,0.08,1.06,1.06,0.57,7,7,18,0.44,0.44,0.60


In [46]:
# Save results to CSV
results_df.to_csv('batch_result.csv', index=False)